In [1]:
# ==========================================
# 1. IMPORT LIBRARY & SETUP (LOCAL)
# ==========================================
import os
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')          # backend non-GUI agar aman saat loop banyak gambar
import matplotlib.pyplot as plt

In [2]:
# --- PATH SETUP (RELATIF) ---
# Asumsi script dijalankan dari root folder project (sejajar dengan folder 'Berhasil')
BASE_DIR      = os.path.join("Berhasil", "Datasets")
FOLDER_IMAGES = os.path.join(BASE_DIR, "Images")
FOLDER_MASKS  = os.path.join(BASE_DIR, "Masks")

OUTPUT_BASE = "Berhasil"
OUTPUT_P1   = os.path.join(OUTPUT_BASE, "Percobaan_1")
OUTPUT_P2   = os.path.join(OUTPUT_BASE, "Percobaan_2")
OUTPUT_P3   = os.path.join(OUTPUT_BASE, "Percobaan_3")

# --- FOLDER BARU UNTUK HASIL SEGMENTASI (HANYA SEGMENTASI) ---
FOLDER_HASIL_SEG = os.path.join(OUTPUT_BASE, "Hasil_Segmentasi")
SUB_MASK     = os.path.join(FOLDER_HASIL_SEG, "Mask_Biner")
SUB_SEG_OBJ  = os.path.join(FOLDER_HASIL_SEG, "Objek_RGB")

for folder in [OUTPUT_P1, OUTPUT_P2, OUTPUT_P3, FOLDER_HASIL_SEG, SUB_MASK, SUB_SEG_OBJ]:
    os.makedirs(folder, exist_ok=True)

assert os.path.isdir(FOLDER_IMAGES), f"Folder tidak ditemukan: {FOLDER_IMAGES}"
assert os.path.isdir(FOLDER_MASKS),  f"Folder tidak ditemukan: {FOLDER_MASKS}"

In [3]:
# ==========================================
# 2. UTILITY FUNCTIONS (Dari file .txt kamu)
# ==========================================
def load_image(filename):
    path_img  = os.path.join(FOLDER_IMAGES, filename)
    path_gt   = os.path.join(FOLDER_MASKS, filename)

    img_bgr = cv2.imread(path_img)
    if img_bgr is None:
        raise FileNotFoundError(f"Gambar tidak ditemukan: {path_img}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    mask_gt = cv2.imread(path_gt, cv2.IMREAD_GRAYSCALE)
    return img_bgr, img_rgb, gray, mask_gt

def detect_black_border(gray, thresh_dark=15):
    dark = (gray <= thresh_dark).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(dark, 8)

    h, w = gray.shape
    border_mask = np.zeros_like(gray)
    for i in range(1, n):
        x, y, bw, bh, _ = stats[i]
        if x == 0 or y == 0 or x + bw >= w or y + bh >= h:
            border_mask[labels == i] = 255
    return border_mask == 0

def otsu_on_valid_area(gray, valid_mask):
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    valid_px = blurred[valid_mask].reshape(-1, 1)

    thr_val, _ = cv2.threshold(valid_px, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    thresh = np.zeros_like(gray)
    thresh[(blurred <= thr_val) & valid_mask] = 255
    return blurred, thresh, thr_val

def evaluate_mask(pred, gt):
    if gt is None:
        return None
    p = (pred > 127).astype(np.uint8)
    g = (gt   > 127).astype(np.uint8)

    inter = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()

    iou  = inter / union if union > 0 else 0.0
    dice = 2 * inter / (p.sum() + g.sum()) if (p.sum() + g.sum()) > 0 else 0.0
    prec = inter / p.sum() if p.sum() > 0 else 0.0
    rec  = inter / g.sum() if g.sum() > 0 else 0.0
    return {"IoU": iou, "Dice": dice, "Precision": prec, "Recall": rec}

In [4]:
# ==========================================
# 3. PERCOBAAN 1 — OTSU THRESHOLDING
# ==========================================
kernel = np.ones((5, 5), np.uint8)
files = sorted(os.listdir(FOLDER_IMAGES))
all_metrics_p1 = []

threshold_akurasi = 0.7
jumlah_lolos = {"Otsu" : 0, "Otsu_Morph" : 0, "Total" : 0}
akurasi_otsu = []
akurasi_otsu_morph = []

print(f"Memproses {len(files)} gambar...")

for nama_file in files:
    stem = os.path.splitext(nama_file)[0]
    
    # --- LOAD ---
    img_bgr, img_rgb, gray, mask_gt = load_image(nama_file)

    # --- PROSES ---
    valid_mask = detect_black_border(gray)
    blurred, thresh, thr_val = otsu_on_valid_area(gray, valid_mask)
    
    mask_inverted = np.where(valid_mask == 1, 255 - thresh, thresh)
    
    # Morfologi
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel)
    closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    segmentation_mask = cv2.bitwise_not(closing)
    segmentation_mask[~valid_mask] = 0
    
    # Objek Tersegmentasi (RGB)
    segmented = cv2.bitwise_and(img_rgb, img_rgb, mask=segmentation_mask)

    # --- SIMPAN FILE TERPISAH (HANYA SEGMENTASI) ---
    
    # 1. Simpan Mask Biner
    path_mask = os.path.join(SUB_MASK, f"{stem}_mask.png")
    cv2.imwrite(path_mask, segmentation_mask)

    # 2. Simpan Objek Tersegmentasi (RGB di atas background hitam)
    path_seg = os.path.join(SUB_SEG_OBJ, f"{stem}_seg.png")
    cv2.imwrite(path_seg, cv2.cvtColor(segmented, cv2.COLOR_RGB2BGR))

    # --- EVALUASI ---
    m = evaluate_mask(segmentation_mask, mask_gt)
    all_metrics_p1.append(m)

    # Cek Akurasi
    if np.mean(mask_inverted==mask_gt) > threshold_akurasi:
        jumlah_lolos["Otsu"] += 1
    if np.mean(segmentation_mask==mask_gt) > threshold_akurasi:
        jumlah_lolos["Otsu_Morph"] += 1
    jumlah_lolos["Total"] += 1
    akurasi_otsu.append(np.mean(mask_inverted==mask_gt))
    akurasi_otsu_morph.append(np.mean(segmentation_mask==mask_gt))

# --- RINGKASAN RATA-RATA ---
keys = ["IoU", "Dice", "Precision", "Recall"]
print("\n=== Rata-rata Metrik Percobaan 1 (Otsu) ===")
for k in keys:
    vals = [m[k] for m in all_metrics_p1 if m]
    print(f"  {k:10s}: {np.mean(vals):.4f}")

print(f"\nAkurasi Berdasarkan Jumlah Gambar (threshold = {threshold_akurasi}) :")
print(f"Otsu\t\t: {jumlah_lolos['Otsu']/jumlah_lolos['Total']:.4f}")
print(f"Otsu_Morph\t: {jumlah_lolos['Otsu_Morph']/jumlah_lolos['Total']:.4f}")
print(f"\nRata2 Akurasi :")
print(f"Otsu\t\t: {np.mean(akurasi_otsu):.4f}")
print(f"Otsu_Morph\t: {np.mean(akurasi_otsu_morph):.4f}")

print(f"\nHasil segmentasi (Mask & Objek RGB) telah disimpan di: {FOLDER_HASIL_SEG}")

Memproses 25 gambar...

=== Rata-rata Metrik Percobaan 1 (Otsu) ===
  IoU       : 0.5517
  Dice      : 0.6958
  Precision : 0.6579
  Recall    : 0.8109

Akurasi Berdasarkan Jumlah Gambar (threshold = 0.7) :
Otsu		: 0.9600
Otsu_Morph	: 0.9600

Rata2 Akurasi :
Otsu		: 0.8366
Otsu_Morph	: 0.8416

Hasil segmentasi (Mask & Objek RGB) telah disimpan di: Berhasil\Hasil_Segmentasi
